# Estudo de ML — Bakehouse Sales Transactions

**Dataset:** `samples.bakehouse.sales_transactions` — 3.333 transações de maio/2024 de franquias de padaria simuladas.

**Colunas:** `transactionID`, `customerID`, `franchiseID`, `dateTime`, `product`, `quantity`, `unitPrice`, `totalPrice`, `paymentMethod`, `cardNumber`

**Objetivo deste notebook:**
1. Explorar os dados (EDA)
2. Fazer engenharia de features
3. Treinar um modelo de regressão (Spark ML) para prever `totalPrice`
4. Avaliar e registrar o experimento no MLflow

> **Nota didática:** em transações reais `totalPrice = quantity * unitPrice` é praticamente determinístico, então o modelo deve acertar quase tudo. Como exercício, troque o alvo para `quantity` (prever quantidade vendida por produto/franja/horário) — aí sim você tem um problema de ML interessante com dados sintéticos.

In [0]:
# Carregar a tabela como DataFrame Spark
df = spark.table("samples.bakehouse.sales_transactions")

print(f"Linhas: {df.count()}")
print(f"Colunas: {len(df.columns)}")
display(df.limit(10))

In [0]:
# Estatísticas básicas das colunas numéricas
display(df.summary())

# Valores distintos das colunas categóricas
df.select("product").distinct().orderBy("product").show(truncate=False)
df.select("paymentMethod").distinct().show(truncate=False)

In [0]:
# Vendas por produto
produtos = spark.sql("""
  SELECT product, COUNT(*) AS transacoes, SUM(quantity) AS unidades,
         ROUND(SUM(totalPrice), 2) AS receita_total
  FROM samples.bakehouse.sales_transactions
  GROUP BY product
  ORDER BY receita_total DESC
""")
display(produtos)

# Vendas por dia
vendas_dia = spark.sql("""
  SELECT DATE(dateTime) AS dia, COUNT(*) AS transacoes,
         ROUND(SUM(totalPrice), 2) AS receita
  FROM samples.bakehouse.sales_transactions
  GROUP BY dia
  ORDER BY dia
""")
display(vendas_dia)

## Engenharia de features

Criamos features a partir de `dateTime` (hora, dia da semana, fim de semana, faixa horária) e codificamos as categóricas (`product`, `paymentMethod`) com `StringIndexer` + `OneHotEncoder` — o padrão do Spark ML para variáveis categóricas.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql import functions as F

# Features temporais a partir do dateTime
df_feat = (df
    .withColumn("hora", F.hour("dateTime"))
    .withColumn("dia_semana", F.dayofweek("dateTime"))
    .withColumn("fim_de_semana", F.when(F.dayofweek("dateTime").isin(1, 7), 1).otherwise(0))
    .withColumn("faixa_horaria", F.floor(F.col("hora") / 4))
)

cat_cols = ["product", "paymentMethod"]
num_cols = ["quantity", "unitPrice", "franchiseID", "hora", "dia_semana", "fim_de_semana", "faixa_horaria"]

# Indexers + encoders para cada categórica
stages = []
for c in cat_cols:
    indexer = StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    stages += [indexer, encoder]

assembler = VectorAssembler(
    inputCols=num_cols + [f"{c}_ohe" for c in cat_cols],
    outputCol="features",
)
stages.append(assembler)

display(df_feat.select("transactionID", "dateTime", "hora", "dia_semana", "product", "quantity", "unitPrice", "totalPrice").limit(10))

## Treinamento

Dividimos em treino/teste (80/20) e treinamos um `RandomForestRegressor`. O `Pipeline` do Spark garante que as mesmas transformações (indexação, one-hot, assembler) sejam aplicadas igualmente no treino e na inferência — essencial para evitar *data leakage* em produção.

In [0]:
from pyspark.ml.regression import RandomForestRegressor

train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestRegressor(featuresCol="features", labelCol="totalPrice", numTrees=100, seed=42)
stages.append(rf)

pipeline = Pipeline(stages=stages)
model = pipeline.fit(train_df)
print("Modelo treinado com sucesso")

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

preds = model.transform(test_df)

# Métricas de regressão
evaluator = RegressionEvaluator(labelCol="totalPrice", predictionCol="prediction")

rmse = evaluator.setMetricName("rmse").evaluate(preds)
r2 = evaluator.setMetricName("r2").evaluate(preds)
mae = evaluator.setMetricName("mae").evaluate(preds)

print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R²:   {r2:.4f}")

display(preds.select("quantity", "unitPrice", "totalPrice", F.round("prediction", 2).alias("predicao"), "product", "paymentMethod").limit(20))

In [0]:
import pandas as pd

# Importância de cada feature no RandomForest
# (colunas one-hot aparecem por posição, pois o SparkML não nomeia os blocos do vetor)
rf_model = model.stages[-1]

feat_importance = pd.DataFrame({
    "posicao_feature": range(len(rf_model.featureImportances.toArray())),
    "importancia": rf_model.featureImportances.toArray(),
})
display(feat_importance.sort_values("importancia", ascending=False))

## Registro no MLflow

O MLflow é a plataforma de rastreamento de experimentos do Databricks. Depois de rodar a célula abaixo, veja o experimento clicando no ícone de **Experiments** na barra lateral do notebook, ou em **Experiments** no menu lateral do workspace.

In [0]:
import mlflow

with mlflow.start_run(run_name="rf_totalprice_bakehouse"):
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_param("test_size", test_df.count())
    mlflow.spark.log_model(model, "pipeline_model")

print("Run registrada no MLflow")

## Próximos passos para estudar

1. **Troque o alvo:** prever `quantity` (regressão mais desafiadora) ou `paymentMethod` (classificação — use `RandomForestClassifier`)
2. **Cross-validation:** `CrossValidator` + `ParamGridBuilder` para tuning de hiperparâmetros
3. **Model Serving:** registre o modelo no Model Registry e publique num Model Serving endpoint
4. **Outras tabelas:** junte com `samples.bakehouse.sales_customers` e `sales_franchises` para enriquecer as features
5. **Séries temporais:** agregue por dia e experimente previsão de receita diária por franquia